In [27]:
import pandas as pd
import numpy as np
import pickle
import time
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

In [28]:
from pathlib import Path

# backend/
BASE_DIR = Path.cwd().parent.parent

# data/raw
DATA_RAW = BASE_DIR / "data" / "raw"

# data/processed
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# tạo folder nếu chưa có
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(DATA_RAW)
print(DATA_PROCESSED)
print("Bắt đầu xử lý dữ liệu...\n")
print(f"Working directory: {BASE_DIR}\n")

e:\multiproject\multiplatform\travil\recommend\backend\data\raw
e:\multiproject\multiplatform\travil\recommend\backend\data\processed
Bắt đầu xử lý dữ liệu...

Working directory: e:\multiproject\multiplatform\travil\recommend\backend



In [29]:
# loading data
def load_data():
    print(" Đang load travol.csv ...")
    df = pd.read_csv(DATA_RAW / "travol.csv", encoding='utf-8', on_bad_lines='skip')
    
    df = df.rename(columns={
        'A': 'title',
        'URL': 'url',
        'Klook-image-img Image': 'image_url',
        'Klook-image-img Description': 'description',
        'Price': 'price',
        'Score-rate': 'rating',
        'Page-activity-recommend-booked-number': 'booked_count',
        'Reviews': 'review_count',
        'Exp-departure__city': 'location',
    })
    
    print(f" Loaded {len(df)} activities")
    return df

In [30]:
#clean dataa
def clean_data(df):
    print(" clean data...")
    cols = ['title', 'url', 'image_url', 'description', 'price', 'rating', 
            'booked_count', 'review_count', 'location']
    df = df[cols].copy()
    
    # Price -> string
    df['price'] = pd.to_numeric(df['price'].astype(str).str.replace(r'US\$|,', '', regex=True), errors='coerce')
    
    # Rating
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(4.5) 
    #thiếu -> mặt định 4.5
    
    df = df.reset_index(drop=True)
    df['product_id'] = df.index
    
    df['combined_text'] = df['title'].fillna('') + " " + \
                          df['description'].fillna('') + " " + \
                          df['location'].fillna('')
    
    print(f" Cleaned activities: {df.shape}")
    return df

#GEOCODING
def add_geolocation(df):
    print("Đang lấy tọa độ GPS (lat, lng)...")
    geolocator = Nominatim(user_agent="klook_recommendation_app", timeout=10)
    
    latitudes = []
    longitudes = []
    
    for i, loc in enumerate(df['location']):
        if pd.isna(loc) or str(loc).strip() in ['', 'nan', 'NaN']:
            latitudes.append(None)
            longitudes.append(None)
            continue
            
        loc_str = str(loc).strip()
        
        try:
            # Logic thông minh theo quốc gia
            lower_loc = loc_str.lower()
            if any(x in lower_loc for x in ['da nang', 'hoi an', 'hue', 'vietnam', 'ba na', 'son tra', 'marble mountain']):
                query = loc_str + ", Vietnam"
            else:
                query = loc_str
                
            location = geolocator.geocode(query)
            
            if location:
                latitudes.append(location.latitude)
                longitudes.append(location.longitude)
            else:
                latitudes.append(None)
                longitudes.append(None)
                
            time.sleep(1.1)
            
        except Exception:
            latitudes.append(None)
            longitudes.append(None)
            time.sleep(1)
    
    df['lat'] = latitudes
    df['lng'] = longitudes
    
    success_rate = df['lat'].notna().sum() / len(df) * 100
    print(f"Geocoding hoàn thành: {df['lat'].notna().sum()}/{len(df)} ({success_rate:.1f}%)")
    return df

In [31]:
#conten-base
def build_content_features(df):
    print("Đang xây dựng TF-IDF & Cosine Similarity...")
    tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
    tfidf_matrix = tfidf.fit_transform(df['combined_text'])
    cosine_sim = cosine_similarity(tfidf_matrix)
    
    with open(DATA_PROCESSED / 'tfidf_vectorizer.pkl', 'wb') as f:
        pickle.dump(tfidf, f)
    np.save(DATA_PROCESSED / 'cosine_sim.npy', cosine_sim)
    
    print(f"Cosine similarity matrix shape: {cosine_sim.shape}")
    return cosine_sim

In [32]:
def extract_real_ratings(activities_df):
    print("Đang extract rating thực từ trareview.xlsx...")
    reviews = pd.read_excel(DATA_RAW / "trareview.xlsx")
    
    reviews = reviews.dropna(subset=['rating', 'link']).copy()
    reviews['rating'] = pd.to_numeric(reviews['rating'], errors='coerce')
    
    # Map link với url activity
    merged = reviews.merge(
        activities_df[['url', 'product_id', 'title']], 
        left_on='link', 
        right_on='url', 
        how='inner'
    )
    
    ratings = merged[['nameuser', 'product_id', 'rating']].copy()
    ratings = ratings.rename(columns={'nameuser': 'user_id'})
    ratings = ratings.dropna(subset=['user_id', 'rating'])
    
    ratings.to_csv(DATA_PROCESSED / 'user_item_ratings_real.csv', index=False)
    
    print(f" Extracted {len(ratings):,} real user ratings")
    print(f"   Users: {ratings['user_id'].nunique()} | Products: {ratings['product_id'].nunique()}")
    return ratings

In [33]:
def main():
    # Load & Clean
    df = load_data()
    df_clean = clean_data(df)
    
    # Geocoding
    df_geo = add_geolocation(df_clean)
    
    # Save activities
    df_geo.to_csv(DATA_PROCESSED / 'products_clean.csv', index=False, encoding='utf-8')
    df_geo.to_parquet(DATA_PROCESSED / 'products_clean.parquet', index=False)
    
    # Content-based
    build_content_features(df_geo)
    
    # Real ratings
    extract_real_ratings(df_geo)
    
    print("\n HOÀN THÀNH TOÀN BỘ ETL!")
    print(f"Dữ liệu đã được lưu tại: {DATA_PROCESSED}")

if __name__ == "__main__":
    main()

 Đang load travol.csv ...
 Loaded 9886 activities
 clean data...
 Cleaned activities: (9886, 11)
Đang lấy tọa độ GPS (lat, lng)...
Geocoding hoàn thành: 3221/9886 (32.6%)
Đang xây dựng TF-IDF & Cosine Similarity...
Cosine similarity matrix shape: (9886, 9886)
Đang extract rating thực từ trareview.xlsx...
 Extracted 26,515 real user ratings
   Users: 11684 | Products: 564

 HOÀN THÀNH TOÀN BỘ ETL!
Dữ liệu đã được lưu tại: e:\multiproject\multiplatform\travil\recommend\backend\data\processed
